# M1.S4 - Resource management and performance metrics
## From requesting resources to measuring useful performance

A shared supercomputer needs two things:

1. a way to decide **who gets which resources and when**;
2. a way to decide **whether those resources are being used effectively**.

In this notebook you will work with both.

### What you will practice

By the end you should be able to:

1. identify the resources an HPC job requests;
2. inspect SciTech partitions and current allocations;
3. create and submit a real Slurm batch job;
4. monitor a job and read its output and accounting information;
5. explain why a job can remain PENDING;
6. distinguish nodes, tasks, CPUs per task, memory and wall time;
7. calculate speedup and parallel efficiency;
8. explain the difference between utilization and useful performance;
9. distinguish peak and sustained performance;
10. choose an appropriate benchmark for a workload.

Use the same cycle throughout:

> **PREDICT -> RUN -> OBSERVE -> EXPLAIN**

## 1 - Who gets the machine?

Imagine a shared system with 100 GPUs and many users.

Three jobs arrive:

| Job | Request | Estimated time |
|---|---:|---:|
| A | 64 GPUs | 8 hours |
| B | 4 GPUs | 30 minutes |
| C | 1 GPU | urgent experiment |

### Predict

Who should run first?

There is no single answer from the information above.

A real scheduler also considers:

- resources currently available;
- priority and fair share;
- requested wall time;
- dependencies;
- partition rules;
- whether a smaller job can fit without delaying higher-priority work.

<details>
<summary><strong>Show explanation</strong></summary>

The scheduler is not simply a FIFO queue.

Its job is to balance **fairness, priority and utilization** while respecting the resource requests of every job.

</details>

## 2 - What resources does a job request?

A job needs more than "a computer".

Typical Slurm resources are:

- **nodes** - machines allocated to the job;
- **tasks** - processes or independent execution units;
- **CPUs per task** - CPU cores available to each task;
- **memory** - RAM available to the job;
- **GPUs** - accelerators when needed;
- **partition** - the class of resources to use;
- **time limit** - maximum wall-clock time.

### Quick check

A program needs:

- 1 node;
- 1 process;
- 4 CPU cores;
- 8 GB RAM;
- about 10 minutes;
- no GPU.

What should you request?

<details>
<summary><strong>Show explanation</strong></summary>

A sensible request would be similar to:

```bash
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:10:00
```

Request enough resources to run efficiently, but no more than you need.

</details>

## 3 - Inspect the real SciTech scheduler

Before submitting anything, inspect the cluster.

We will use:

- `sinfo` - what resources and partitions exist?
- `squeue` - what jobs are currently queued or running?
- Slurm environment variables - what does this Jupyter session itself have?

In [ ]:
import os
import re
import time
import math
import subprocess
from pathlib import Path

MEM_VARS = ["SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"]

def clean_slurm_env():
    env = os.environ.copy()
    for key in MEM_VARS:
        env.pop(key, None)
    return env

def run_command(command):
    print("$", command)
    p = subprocess.run(
        command,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    if p.stdout.strip():
        print(p.stdout.strip())
    if p.stderr.strip():
        print("[stderr]")
        print(p.stderr.strip())
    print("return code:", p.returncode)
    print()
    return p

def submit_slurm(script_path):
    p = subprocess.run(
        ["sbatch", "--parsable", str(script_path)],
        capture_output=True,
        text=True,
        env=clean_slurm_env()
    )
    if p.returncode != 0:
        raise RuntimeError(p.stderr.strip() or "sbatch failed")
    job_id = p.stdout.strip().split(";")[0]
    print("Submitted Slurm job:", job_id)
    return job_id

def slurm_status(job_id):
    if not job_id:
        print("No job ID available.")
        return
    run_command(
        f"squeue -j {job_id} -o '%.10i %.10T %.10M %.5D %R'"
    )

def wait_for_job(job_id, timeout=240, poll=2):
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job_id), "-o", "%T"],
            capture_output=True, text=True
        )
        state = p.stdout.strip()
        if not state:
            print("Job", job_id, "has left squeue.")
            return True
        print("Job", job_id, "state:", state)
        time.sleep(poll)
    print("Timed out while waiting. The job may still be queued or running.")
    return False

def show_output(pattern):
    matches = sorted(Path(".").glob(pattern))
    if not matches:
        print("No matching output file yet:", pattern)
        return
    path = matches[-1]
    print("Output file:", path)
    print(path.read_text(errors="replace"))

print("M1.S4 helper functions ready.")

In [ ]:
run_command("sinfo -a -N -o '%N %P %t %c %m %G'")
run_command("squeue -u $USER -o '%.10i %.16j %.10T %.10M %.5D %R'")

In [ ]:
print("Current Jupyter allocation:")
for key in [
    "SLURM_JOB_ID",
    "SLURM_JOB_PARTITION",
    "SLURM_NODELIST",
    "SLURM_CPUS_PER_TASK",
    "SLURM_MEM_PER_CPU",
    "SLURM_MEM_PER_NODE",
]:
    print(f"{key}={os.environ.get(key, 'not set')}")

### Observe

Answer from the real output:

1. Which partitions are available?
2. Which nodes are visible?
3. Which nodes advertise GPUs?
4. What partition is your Jupyter session using?
5. How many CPUs are allocated to this Jupyter session?

### Important

The node may expose hundreds of logical CPUs, but your current job owns only the resources Slurm allocated to it.

**Visible hardware is not the same as allocated hardware.**

## 4 - The life of a batch job

A typical batch workflow is:

```text
WRITE
  |
SUBMIT with sbatch
  |
PENDING in queue
  |
RUNNING on allocated resources
  |
COMPLETED
  |
OUTPUT + ACCOUNTING
```

Submit does **not** mean "run immediately".

PENDING is often a normal state.

## 5 - Create your first scheduled job

This job asks Slurm for:

- CPU partition;
- 1 task;
- 2 CPUs for that task;
- 1 GB RAM;
- 2 minutes maximum;
- a short computation.

The 15-second sleep gives you a chance to catch the job in `squeue`.

In [ ]:
first_job_script = r'''#!/bin/bash
#SBATCH --job-name=m1s4_first
#SBATCH --partition=cpu
#SBATCH --output=m1s4_first_%j.out
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=1G
#SBATCH --time=00:02:00

echo "=== HPC JOB PASSPORT ==="
echo "JOB_ID=$SLURM_JOB_ID"
echo "NODE=$(hostname)"
echo "PARTITION=$SLURM_JOB_PARTITION"
echo "NODES=$SLURM_JOB_NUM_NODES"
echo "TASKS=$SLURM_NTASKS"
echo "CPUS_PER_TASK=$SLURM_CPUS_PER_TASK"
echo "START=$(date -Is)"

sleep 15

python3 - <<'PY'
n = 100000
result = sum(i*i for i in range(1, n+1))
print("CHECKSUM=", result % 1000003)
PY

echo "RUNNING_UNDER_SLURM=YES"
echo "END=$(date -Is)"
'''

Path("m1s4_first_job.sbatch").write_text(first_job_script)
print(Path("m1s4_first_job.sbatch").read_text())

### Predict before submitting

What do you expect?

- Will `sbatch` execute the script immediately?
- Which node will the scheduler choose?
- Will asking for 2 CPUs make the Python calculation automatically use 2 CPUs?

Write your prediction before running the next cell.

In [ ]:
M1S4_JOB_ID = submit_slurm("m1s4_first_job.sbatch")

## 6 - Monitor the job

Run this cell while the job is pending or running.

Look at:

- job state;
- elapsed time;
- number of nodes;
- `NODELIST(REASON)`.

For a PENDING job, the final column often explains why it has not started.

In [ ]:
slurm_status(globals().get("M1S4_JOB_ID"))

## 7 - Read the result

If the job has already disappeared from `squeue`, that usually means it has finished.

Wait for completion and inspect the output.

In [ ]:
wait_for_job(globals().get("M1S4_JOB_ID"), timeout=240)
show_output(f"m1s4_first_{globals().get('M1S4_JOB_ID', 'NONE')}.out")

### Explain

The output is evidence of what Slurm actually did.

You should be able to identify:

- a unique job ID;
- the compute node selected;
- the partition;
- the number of nodes;
- the number of tasks;
- CPUs per task;
- a correct checksum.

### Key question

We requested 2 CPUs. Did the Python calculation automatically use both?

<details>
<summary><strong>Show explanation</strong></summary>

No.

Slurm **allocates resources**. It does not rewrite a serial program into a parallel program.

A serial Python calculation will normally use one CPU even if the job has been allocated more.

This distinction is fundamental:

> **Resource allocation is not the same thing as parallel execution.**

</details>

## 8 - Inspect accounting information

`squeue` is mainly for current jobs.

After a job finishes, `sacct` can show its history, state, elapsed time and resource allocation.

In [ ]:
job_id = globals().get("M1S4_JOB_ID")
if job_id:
    run_command(
        f"sacct -j {job_id} "
        "--format=JobID,JobName%20,State,Elapsed,AllocCPUS,ReqMem,ExitCode "
        "-n -P"
    )
else:
    print("Run the submission cell first.")

## 9 - Why can a job be PENDING?

A job may wait because of:

- **Resources** - enough CPUs, GPUs or memory are not free;
- **Priority / fair share** - other jobs currently rank higher;
- **Time** - the requested wall time does not fit an available scheduling window;
- **Dependencies** - another job must finish first;
- **Partition / constraints** - the requested resource type is not currently available.

### Predict

Which statement is correct?

A. PENDING means the script is broken.  
B. PENDING means the scheduler cannot or should not start the job yet.  
C. PENDING means the job is already running.

<details>
<summary><strong>Show explanation</strong></summary>

**B.**

A pending job is waiting for the scheduler to find the right resources and scheduling opportunity.

</details>

## 10 - Backfilling: the queue is not simply FIFO

Suppose 6 nodes are free:

| Job | Nodes | Time | Priority |
|---|---:|---:|---|
| A | 8 | 4 h | normal |
| B | 2 | 20 min | normal |
| C | 4 | 1 h | high |
| D | 1 | 10 min | normal |

### Think first

- Which jobs can start now?
- Which one has the strongest priority claim?
- Could another smaller job run at the same time?
- Why is an accurate wall-time request useful?

<details>
<summary><strong>Show explanation</strong></summary>

Job A cannot start because it needs 8 nodes and only 6 are free.

Job C can start because it needs 4 nodes and has high priority.

That leaves 2 nodes, so Job B or Job D may also fit depending on scheduler policy and reservations.

This is the idea behind **backfilling**: a smaller job may run earlier if doing so does not delay a reserved higher-priority job.

Accurate time requests help the scheduler decide whether a job safely fits into a scheduling gap.

</details>

## 11 - A resource request is a contract

Too little:

- the job may fail;
- it may run out of memory;
- it may hit the time limit.

Too much:

- resources sit idle;
- the job may wait longer;
- other users cannot use those resources.

### Classify each request

1. A serial program asks for 64 CPUs "just in case".
2. A program needs 12 GB but asks for 8 GB.
3. A 3-minute test asks for a 24-hour wall time.
4. A GPU program asks for the CPU partition.

<details>
<summary><strong>Show explanation</strong></summary>

1. Over-requested CPUs.  
2. Under-requested memory.  
3. Excessive wall-time request.  
4. Wrong resource/partition request.

</details>

## 12 - Performance is not one number

After resources are allocated, we need to ask whether they are being used well.

Useful metrics include:

| Metric | Question |
|---|---|
| **FLOPS** | How much floating-point work is performed per second? |
| **Bandwidth** | How much data moves per second? |
| **Latency** | How long does an operation or transfer take? |
| **Throughput** | How much work is completed over time? |
| **Utilization** | How much of the available hardware is busy? |
| **Speedup** | How much faster did the program become? |
| **Efficiency** | How effectively are extra parallel resources used? |
| **Sustained performance** | What does the system achieve in a real workload? |

### Important

A system can have high utilization and still perform poorly.

Processors can be busy while waiting on memory, communication, synchronization or inefficient work.

## 13 - Calculate speedup and efficiency

For a fixed problem:

```text
Speedup(p) = T1 / Tp

Efficiency(p) = Speedup(p) / p
```

Example:

- 1 core: 100 s
- 4 cores: 30 s

Predict the speedup and efficiency before running the cell.

In [ ]:
T1 = 100.0
Tp = 30.0
p = 4

speedup = T1 / Tp
efficiency = speedup / p

print(f"Speedup:    {speedup:.2f}x")
print(f"Efficiency: {efficiency:.2%}")

### Explain

The program is about **3.33x faster**, not 4x faster.

Parallel efficiency is about **83%**.

The missing performance can come from:

- serial work;
- communication;
- synchronization;
- load imbalance;
- data movement;
- parallel-management overhead.

## 14 - More allocated CPUs do not automatically mean more performance

Now test that idea on the real cluster.

We will run the **same serial Python workload** twice:

1. one job allocated 1 CPU;
2. one job allocated 4 CPUs.

The code itself remains serial.

### Prediction

Will the 4-CPU allocation make the serial program four times faster?

In [ ]:
serial_work = r'''import time

N = 12000000

t0 = time.perf_counter()
checksum = 0
for i in range(N):
    checksum += (i * i) % 97
elapsed = time.perf_counter() - t0

print(f"ELAPSED_SEC={elapsed:.6f}")
print(f"CHECKSUM={checksum}")
'''

Path("m1s4_serial_work.py").write_text(serial_work)

def make_serial_job(cpus):
    text = f'''#!/bin/bash
#SBATCH --job-name=m1s4_c{cpus}
#SBATCH --partition=cpu
#SBATCH --output=m1s4_c{cpus}_%j.out
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task={cpus}
#SBATCH --mem=1G
#SBATCH --time=00:03:00

echo "ALLOC_CPUS=$SLURM_CPUS_PER_TASK"
echo "NODE=$(hostname)"
python3 m1s4_serial_work.py
'''
    path = Path(f"m1s4_serial_c{cpus}.sbatch")
    path.write_text(text)
    return path

job1_path = make_serial_job(1)
job4_path = make_serial_job(4)

print(job1_path.read_text())
print(job4_path.read_text())

Run the next cell once.

It submits the 1-CPU job, waits for it, then submits the 4-CPU job. Running them sequentially reduces interference between the two measurements.

In [ ]:
M1S4_SERIAL_1 = submit_slurm("m1s4_serial_c1.sbatch")
wait_for_job(M1S4_SERIAL_1, timeout=240)
show_output(f"m1s4_c1_{M1S4_SERIAL_1}.out")

M1S4_SERIAL_4 = submit_slurm("m1s4_serial_c4.sbatch")
wait_for_job(M1S4_SERIAL_4, timeout=240)
show_output(f"m1s4_c4_{M1S4_SERIAL_4}.out")

In [ ]:
def read_elapsed(path):
    text = Path(path).read_text(errors="replace")
    m = re.search(r"ELAPSED_SEC=([0-9.]+)", text)
    return float(m.group(1)) if m else None

p1 = f"m1s4_c1_{globals().get('M1S4_SERIAL_1', 'NONE')}.out"
p4 = f"m1s4_c4_{globals().get('M1S4_SERIAL_4', 'NONE')}.out"

if Path(p1).exists() and Path(p4).exists():
    t1 = read_elapsed(p1)
    t4 = read_elapsed(p4)
    print(f"1 allocated CPU : {t1:.6f} s")
    print(f"4 allocated CPUs: {t4:.6f} s")
    if t1 and t4:
        apparent_speedup = t1 / t4
        allocation_efficiency = apparent_speedup / 4
        print(f"Apparent speedup: {apparent_speedup:.3f}x")
        print(f"Allocation efficiency relative to 4 CPUs: {allocation_efficiency:.1%}")
else:
    print("Run the two jobs first.")

### Observe and explain

The two runtimes should be in the same general range.

Small differences are normal because the machine is shared and CPU frequency, cache state and system activity vary.

The key result is conceptual:

> **Slurm can allocate 4 CPUs, but a serial program still has to be parallelized before it can use those CPUs productively.**

If the runtime stays roughly the same while four CPUs are reserved, the resource allocation is inefficient.

## 15 - Amdahl's Law: why speedup eventually stops scaling

For a fixed problem, if a fraction `s` of the program is serial:

```text
Speedup(p) = 1 / (s + (1-s)/p)
```

Suppose 20% of the program is serial.

What is the theoretical maximum speedup with infinitely many processors?

<details>
<summary><strong>Show explanation</strong></summary>

As `p -> infinity`, the parallel part approaches zero time:

```text
maximum speedup = 1 / 0.20 = 5x
```

Even unlimited processors cannot remove the serial part.

</details>

In [ ]:
def amdahl_speedup(serial_fraction, processors):
    return 1.0 / (
        serial_fraction + (1.0 - serial_fraction) / processors
    )

serial_fraction = 0.20

for p in [1, 2, 4, 8, 16, 64, 1024]:
    s = amdahl_speedup(serial_fraction, p)
    e = s / p
    print(f"p={p:4d}  speedup={s:6.3f}x  efficiency={e:7.2%}")

print("Theoretical limit:", 1 / serial_fraction, "x")

## 16 - Peak performance vs sustained performance

**Peak performance**

- theoretical maximum under ideal conditions;
- useful for describing hardware capability.

**Sustained performance**

- what a real workload actually achieves;
- includes memory limits, communication, overheads, contention and application behavior.

### Predict

System X has a theoretical peak of 5 PFLOPS but sustains 3 PFLOPS on a workload.

What fraction of peak is sustained?

In [ ]:
peak = 5.0
sustained = 3.0

fraction = sustained / peak

print(f"Sustained / peak = {fraction:.1%}")

## 17 - What is a benchmark?

A benchmark is a standardized test used to measure and compare performance.

A useful benchmark should be:

- **relevant** to the workload or component we care about;
- **portable** across systems;
- **accepted** by a community;
- **comparable** between systems.

A benchmark is meaningful only for the question it was designed to answer.

## 18 - Different benchmarks answer different questions

Match each benchmark to the question it answers best.

| Benchmark | Main focus |
|---|---|
| **HPL** | dense floating-point performance |
| **HPCG** | sparse linear algebra, memory and communication |
| **STREAM** | sustainable memory bandwidth |
| **MLPerf** | AI training and inference |
| **Green500** | performance per watt |

### Choose the benchmark

1. Measure memory bandwidth.
2. Compare dense floating-point throughput.
3. Evaluate sparse scientific workloads.
4. Compare AI training systems.
5. Compare energy efficiency.

<details>
<summary><strong>Show explanation</strong></summary>

1. STREAM  
2. HPL  
3. HPCG  
4. MLPerf  
5. Green500

</details>

## 19 - Benchmark detective

A new system wins an HPL comparison.

Can you conclude that it is the best machine for every workload?

<details>
<summary><strong>Show explanation</strong></summary>

No.

HPL emphasizes dense floating-point computation.

A different workload may be limited by:

- memory bandwidth;
- sparse data access;
- network communication;
- storage;
- AI software and accelerator behavior;
- energy efficiency.

The correct benchmark depends on the performance question.

</details>

## 20 - Mini-practice: design a sensible job request

You have a program with these requirements:

- one process;
- four CPU cores;
- 8 GB RAM;
- about 10 minutes;
- CPU only.

Write the `#SBATCH` lines you would use.

Then answer:

1. Which partition would you request?
2. Which command submits it?
3. Which command monitors it?
4. Which command checks the finished job?
5. What would you inspect if it remains PENDING?

<details>
<summary><strong>Show suggested solution</strong></summary>

```bash
#!/bin/bash
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH --time=00:10:00
#SBATCH --output=myjob_%j.out
```

Useful commands:

```bash
sbatch myjob.sbatch
squeue -u $USER
sacct -j JOBID
```

For a pending job, inspect the state and the `NODELIST(REASON)` field in `squeue`.

</details>

## 21 - Exit ticket

Answer each in one or two sentences.

### 1. What is the difference between allocating CPUs and using CPUs?

Your answer:

### 2. Why can a job be PENDING even if it is correct?

Your answer:

### 3. Why can requesting too many resources be harmful?

Your answer:

### 4. What is the difference between speedup and efficiency?

Your answer:

### 5. Why is HPL not enough to describe every HPC workload?

Your answer:

<details>
<summary><strong>Show explanation</strong></summary>

- **Allocation vs use:** Slurm reserves resources; the application must be written to use them.
- **PENDING:** the scheduler may be waiting for resources, priority, timing, dependencies or constraints.
- **Over-requesting:** resources can sit idle, waiting time can increase and other users lose access.
- **Speedup vs efficiency:** speedup compares runtime improvement; efficiency divides that speedup by the number of resources used.
- **HPL:** it emphasizes dense floating-point computation and does not represent every memory, network, sparse, AI or energy workload.

</details>

## What you should leave with

- HPC resources are **shared**, so access must be scheduled.
- A job requests **nodes, tasks, CPUs, memory, accelerators and time**.
- `sbatch`, `squeue`, `sacct`, `sinfo`, `srun`, `salloc` and `scancel` cover most daily Slurm workflows.
- PENDING is often normal.
- A resource request is a **contract**: ask for what you need, not as much as possible.
- Allocating more CPUs does not automatically parallelize a program.
- Performance needs the **right metric**: FLOPS, bandwidth, latency, throughput, utilization, speedup or efficiency.
- Peak performance and sustained performance are different.
- Different benchmarks answer different questions.

Next: **M1.S5 - Practice 1: connect, run, submit and document your first HPC workflow**.